# Forced Alignment Result

> Standardized word-level forced-alignment DTOs — the data noun forced-alignment tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

**Canonical home (execution stage 8 — Option C / PILLAR 1c):** `ForcedAlignResult` lives here in `cjm-capability-primitives` so a pure-compute forced-alignment tool capability depends only on this data noun, never on the adapter machinery (`TaskAdapter`, the tool protocol, persistence). It relocated here from `cjm-forced-alignment-adapter-interface.core` (itself relocated from `cjm-transcription-plugin-system` at stage 2); the old homes re-export class-identically (REMOVE-AFTER-OVERHAUL) until the cascade retires them. The wire kind is `"forced_alignment.result"`. Because `items` holds typed `ForcedAlignItem` objects, a custom `from_dict` re-types them on wire-decode (the auto flat reconstruct would leave them as dicts).

In [ ]:
#| default_exp forced_alignment

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional

from cjm_plugin_system.core.wire import wire_type

## ForcedAlignItem

A single word-level alignment result from a forced alignment model. Each item represents one word with its temporal boundaries in the audio.

Note that forced alignment models typically strip punctuation from the input text. The consuming service is responsible for mapping these stripped words back to the original punctuated text.

In [ ]:
#| export
@dataclass
class ForcedAlignItem:
    """A single word-level alignment result."""
    text: str        # The aligned word (punctuation typically stripped by model)
    start_time: float  # Start time in seconds
    end_time: float    # End time in seconds

## ForcedAlignResult

Standardized output for all forced alignment capabilities. Contains the list of word-level alignments and optional metadata about the alignment run.

In [ ]:
#| export
@wire_type("forced_alignment.result")
@dataclass
class ForcedAlignResult:
    """Standardized output for all forced alignment capabilities."""
    items: List[ForcedAlignItem]                          # Word-level alignments
    metadata: Dict[str, Any] = field(default_factory=dict)  # Capability-specific metadata

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any],  # Wire payload (the substrate envelope's "data")
    ) -> "ForcedAlignResult":  # Reconstructed result with typed items
        """Reconstruct from a wire payload, re-typing nested items.

        `items` holds typed `ForcedAlignItem` objects, so the substrate's typed
        wire envelope (stage 2) reconstructs them host-side here rather than
        leaving bare dicts (which would break attribute access like `it.text`).
        """
        return cls(
            items=[it if isinstance(it, ForcedAlignItem) else ForcedAlignItem(**it)
                   for it in (d.get("items") or [])],
            metadata=d.get("metadata") or {},
        )

## Testing

In [ ]:
# Test ForcedAlignItem creation + serialization round-trip
item = ForcedAlignItem(text="November", start_time=1.04, end_time=1.6)
assert item.text == "November" and item.start_time == 1.04 and item.end_time == 1.6
d = asdict(item)
assert d == {"text": "November", "start_time": 1.04, "end_time": 1.6}
assert ForcedAlignItem(**d) == item
print("ForcedAlignItem serialization round-trip: OK")

In [ ]:
# Test ForcedAlignResult creation
items = [
    ForcedAlignItem(text="November", start_time=1.04, end_time=1.6),
    ForcedAlignItem(text="the", start_time=1.6, end_time=1.68),
    ForcedAlignItem(text="10th", start_time=1.76, end_time=2.08),
]
result = ForcedAlignResult(items=items, metadata={"model_id": "Qwen/Qwen3-ForcedAligner-0.6B", "language": "English"})
assert len(result.items) == 3 and result.items[0].text == "November"
assert result.metadata["model_id"] == "Qwen/Qwen3-ForcedAligner-0.6B"

In [ ]:
# Test minimal result (empty items, no metadata)
minimal = ForcedAlignResult(items=[])
assert minimal.items == [] and minimal.metadata == {}
print(f"Minimal result: {minimal}")

In [ ]:
# Wire-format executable test: the nested result round-trips TYPED through
# the simulated worker boundary — items come back as ForcedAlignItem, not
# dicts (the custom from_dict's whole job).
import json as _json
from cjm_plugin_system.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

res = ForcedAlignResult(
    items=[ForcedAlignItem(text="one", start_time=0.0, end_time=0.4),
           ForcedAlignItem(text="small", start_time=0.4, end_time=0.9)],
    metadata={"model": "qwen3-fa"},
)
envelope = wire_encode(res)
assert envelope[WIRE_KIND_KEY] == "forced_alignment.result"
back = wire_decode(_json.loads(_json.dumps(envelope)))
assert isinstance(back, ForcedAlignResult) and back == res
assert all(isinstance(it, ForcedAlignItem) for it in back.items)
print("forced_alignment.result wire round-trip OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()